# Merging Train Line footprints for citywide analysis
Based on this [CTA line data](https://data.cityofchicago.org/Transportation/CTA-L-Rail-Lines/xbyr-jnvx/about_data) from the City of Chicago

<hr>

The current CTA line data I have is split into 152 LineString elements. I want my analysis to be robust enough to detect if a given property is close to two distinct train line (and the potential noise effects of that double exposure). With my CTA file, I will turn the small LineStrings into longer stretchs that share a common path and elevation status (Subway/Elevated or At Grade) with each longer stretch having a unique identifier. I also think it would be helpful to have a specific identifier for if the train segment runs in the median of a highway, because noise from those segments (particualrly the Blue line outside of the Loop) might not inherently have as much of an impact as the general highway noise.

I'm using [mapshaper](https://mapshaper.org) to help me interact with the geojson file in a simple interface to identify the ids of line segments I can combine
<div align="center">
    <img src="imgs/cta_1.png" width=40%> <img src="imgs/cta_2.png" width=40%><br>
</div>

In [29]:
import geopandas as gpd

In [30]:
cta_lines = gpd.read_file('data/cta_lines.geojson')
cta_lines = cta_lines.to_crs(epsg=26971)

In [31]:
cta_lines.to_csv('data/cta_lines.csv', index=False)

<hr>

## How I Organized the Lines

I used MapShaper to view each individual line segment and went segment-by-segment to determine what group it should be in (and how it made sense to group them). I ultimately decided to group train lines such that concurrent segments that form one straight line are in the same group, and any perpendicular or connecting lines would be in their own group. The below (rudimentary) illustration shows why:
<br>
<div align="center">
<img src="imgs/train_intersect_problem.png" width=50%>
</div>
<br>
Suppose we are interested in calculating the noise experienced from train lines by the home with the black outline. The single shortest path between that building's centroid and any CTA line would extend to the Pink Line (as indicated by the solid red-colored line between the two elements). In this case, though, my current function would determine that our subject building experiences relatively low noise exposure because of the taller buildings in between it and the train tracks.

However, the building is also near the Green Line, and the shortest path to that line (indicated by the solid blue-colored line) would also find that more noise can reach our target building because there is only one, shorter building inbetween the two elements.

The path to the Green Line is not the second-shortest line, though, and it's not clear how many interceding lines would form a shorter path between the building and any CTA tracks (interceding lines indicated by the dashed red-colored lines).

Thus, my function will:
1. Find the shortest path and conduct any calculations with that line
2. **Record the group ID of that line path** (in this case something like "*pink-eag-curve"*)
3. Look for the next shortest path to **any line other than the one for which I've already reacorded a group ID**

I opened the [.csv version](data/cta_lines.csv) of the CTA lines in Sheets and created my line groups (16 in total). Below, I'm referencing that sheet and each element's unique ":id" to create my grouped line elements.

You can see my specific train groupings in the [my_cta_groups.csv](data/my_cta_groups.csv).

<hr>

In [49]:
cta_lines.head()

,:id,:version,:created_at,:updated_at,lines,description,type,legend,shape_len,geometry
0,row-jfz4-3j8m~bhsb,rv-ucfc-iy3g~j5vu,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Orange, Pink, Purple (Express)",Tower 12 to Library,Elevated or At Grade,ML,647.793224715,"MULTILINESTRING ((358531.229 578671.331, 35858..."
1,row-j8yp-4buv.s655,rv-yw5q_s4wj.dk6g,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Purple",Subway Portal B to Sedgwick,Elevated or At Grade,BR,4303.79492895,"MULTILINESTRING ((357580.312 582384.742, 35754..."
2,row-m5nn~7ffa-s9eu,rv-ky69_kik2~4emh,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,Yellow Line,Oakton-Skokie to Howard,Elevated or At Grade,YL,21195.44443,"MULTILINESTRING ((348540.536 595177.981, 34854..."
3,row-m9xp_txdh-c9wu,rv-hfga~uzwm-4f37,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Green, Orange, Pink, Purple (Exp)",Washington/Wabash to Adams/Wabash,Elevated or At Grade,ML,1352.12055387,"MULTILINESTRING ((358704.1 578961.761, 358696...."
4,row-bf34.b8eu-gbg7,rv-urkc.8hjb_b78p,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Green, Orange, Pink, Purple (Exp)",State/Lake to Washington/Wabash,Elevated or At Grade,ML,1317.60492405,"MULTILINESTRING ((358693.16 579373.743, 358685..."


## Dealing with "Subway" segments
Subways don't generate *zero* noise perceptable above ground, but it is significantly lower as compared to Elevated or At Grade train segments (just from my own personal experiences in Chicago). Because of this, I'll group all the subway segments onto one unified shape – essentially assuming the impact of being by one subway is not significantly different than being by two subways.

Also important to note: the only two trains that go below ground for parts of their runs are the Blue and Red lines, and those two lines are only close to each other for about a 6 block stretch, and that stretch is within the Loop, the place where every single train line is in close proximity. Because if this, too, I think it will be OK to not assume an addative effect of living near both the Blue and Red lines the same way someone would experience living near the separation of the Red and Brown lines on the north side.

In [ ]:
subway_1 = ["row-cczx-39nv~xgkd", "row-fega_z5tv_gjfr", "row-gqic_mup4.6dv6", "row-ex7j-3kgs.ehqm", "row-2vnp-yk2t_xp4t", "row-ctbg-zt9n_wwb8", "row-dtsp.3yjf_96wz", "row-gyfn~5dvt_hw9z", "row-ybu8~kept-dedr", "row-nhgt_eurb.bjzk", "row-dv7d.6jgu_nww3", "row-wue9_xer7.pjgh", "row-7erp_5r65_vahi", "row-4qx7_akzu.5vm4", "row-gpui~wnrf_g38p", "row-bpqp-t4sb-ynp6", "row-i32x.kb5j-nz93", "row-d4t7~6r4y~eaw3", "row-798g_wbnj-qyai", "row-ddcf.x73j.uwqf", "row-knb4-ra48.kdqd"]

cta_lines[]

bl_hwy_north_2 = ["row-dbt4_gayf-b76v", "row-d3zt~rzw6_d826", "row-5hwm_4ubb_va39", "row-svwm_u72e.jf72", "row-i2hs_hv59-jxq2", "row-esip_4f39~3auy", "row-v9nh~365i~h3pc", "row-cb6x~6jdf~k4j5"]

bl_eag_north_3 = ["row-imi6-7rmp_i8ss", "row-7j5e~c5ub.94n7", "row-nyzq-gr9z_z3h4", "row-i9v9.7jfu_ju3v"]

bl_hwy_south_4 = ["row-xjgr-inf4~5uny", "row-2qyt-hb24.6kq9", "row-seb9~cpje.ftkd", "row-2xst.v6rv_ibn9", "row-fyf3_c5sw_te48", "row-gemb-i2ki-pwxa", "row-fh7d-sefj.7baq", "row-dg3r~ferx~v9ic", "row-7qaw~sjyg~asnb", "row-kv8f.2d4v.g68f", "row-v3iq.5u9t.b64b"]

gr_eag_opclinton_5 = ["row-tpny_tnj3.wee2", "row-tfh5-3d98-9dwx", "row-k6u8.gu7r~5azt", "row-4zfx~cgts.mhnk", "row-j32w~cwdn_m9ir", "row-2633~vv6g.wfp5", "row-bpiu_vp9f_tfq6", "row-fhgn_bva3_zcqk", "row-75vs~6as5-3kkj", "row-wwrf_bnv5-bhku", "row-m9nk.cur9~2ygk", "row-e574-bdtn~dqcs", "row-bs3k~3ier-ywze", "row-f8md_thwj~dtin", "row-yhd5~gj3q-wjs3"]

pi_eag_cicclinton_6 = ["row-w9ku_2d8z_yrh7", "row-8dr7-8n3v~3hhw", "row-s2sw_2k26-kzgz", "row-yrni.yzwu_e33c", "row-megz-27eu~skqs", "row-6wiy-54tv.5yds", "row-5a93_9mtb-w2fd", "row-rue7-mkrk-wxck", "row-bpdz.xntj~mjs3", "row-4p2f~dgxb-4sf9", "row-ad5n-a4tx~5hxg"]

or_eag_mdwchinatown_7 = ["row-xaid_v53j~rrh7", "row-q3uh.kqmm.pawu", "row-cfzs_sypg_bgvy", "row-wrzp~phd8.q6m5", "row-sqic_pr2s-mis2", "row-rysk~y9v4_fph5"]

or_mdw_mdwchinatown_8 = ["row-wuwu-7aiv.42fx"]

gr_eag_ashland59th_9 = ["row-37j5_xwai-a8ms", "row-stdn-tgeg-4ew2"]

gr_eag_cotgroloop_10 = ["row-eh57-2r7b.rgpn", "row-2873~jtpd~t3xw", "row-2i5g-jkdm~xfgc", "row-wfnq_37sn.9huh", "row-8trx_ed6x~c6z4", "row-r5wd~7xbp_hf7m", "row-8z97_spdc-qtqs", "row-7c3r.bfrz.bbsh", "row-7icu.dbhm.hexv", "row-e7s8_5bcb~m6x6", "row-539q-zm5a_nbyn", "row-sz5i~2ukk~k3hd"]

br_eag_kimballmerge_11 = ["row-reg8-egn8-pmzi", "row-ctmx~56gv.fxey", "row-a8y4_sej9-wr9p", "row-7pex~2prb-6ahw", "row-ijjw~6sb9_t2qf", "row-7uyg_4feh~227e", "row-ggv3_4xe7.7knc", "row-zdq6_hppu_38nt", "row-s95x.95yx~fkc9", "row-n3xv.afms-yrw6", "row-mszp-nzxx.tknd"]

br_eag_willowloop_12 = ["row-j8yp-4buv.s655", "row-9n5a.ut8h~mg2n", "row-9ww5_zqux~3hvt", "row-pb7c_9wg5.ijn6"]

yl_eag_all_13 = ["row-m5nn~7ffa-s9eu", "row-tb6g~q46j-uv4d"]

rp_eag_north_14 = ["row-rg43_w5du_2e33", "row-x24v.k8bi-5m99", "row-5cp8.f5t4-si53", "row-8q75~t3s6-xsu7", "row-jk2r.cndz-wv55", "row-i4qr.3a7c~w9up", "row-97s2_i8g3.85w6", "row-hqx5.ustb_mj5c", "row-4wj5~dtnx~rpkn", "row-vz5p~idcm~acsy", "row-sdjn-7384~sbdd", "row-adkt~mxk6.4f39", "row-8uzf~dqka_kmt9", "row-ef9f.f9i7.qv4x", "row-vp5m~rr6c.zgey", "row-s7dp~tv78-35bb", "row-ya8a-w6pn-xmdc", "row-ftfp-tttu_6wf3", "row-hfcp~88wu.wvwh", "row-3vwe~cmy3~dxx4", "row-a4qw~y7rn~tjam", "row-82iu.uvx6.xxzt", "row-zjd3~ksai-9bar", "row-m4kr-3wj2.8bb6", "row-enix-mz6s.5dnc", "row-htib_27k7.2zus", "row-j6ha_v2b6.83ka"]

rd_hwy_south_15 = ["row-g9ft-mmeq-9hs7", "row-y2bw_bjf3.999d", "row-g7d6-tvxy~vy8a", "row-4hv9_kzkq-n9tp", "row-4u8p_8yze-73k4", "row-4eci-b6ea~vsip", "row-jq9n.u6ja-p8c6", "row-22zh-ptsa~sgfk"]

all_eag_loop_16 = ["row-m9xp_txdh-c9wu", "row-bf34.b8eu-gbg7", "row-pqwc~hzjx~fynh", "row-dav3-aiv4~hbi4", "row-ddmx-tcxy~s3i7", "row-jfz4-3j8m~bhsb", "row-7xts_52qy.nia8", "row-esej_3d76_hwdq", "row-va89_mwps~cmb8", "row-ywdj.2r7d_kbnh"]

groups = ["subway_1", "bl_hwy_north_2", "bl_eag_north_3", "bl_hwy_south_4", "gr_eag_opclinton_5", "pi_eag_cicclinton_6", "or_eag_mdwchinatown_7", "or_mdw_mdwchinatown_8", "gr_eag_ashland59th_9", "gr_eag_cotgroloop_10", "br_eag_kimballmerge_11", "br_eag_willowloop_12", "yl_eag_all_13", "rp_eag_north_14", "rd_hwy_south_15", "all_eag_loop_16"]

In [45]:
cta_lines.dtypes

:id                         object
:version                    object
:created_at    datetime64[ms, UTC]
:updated_at    datetime64[ms, UTC]
lines                       object
description                 object
type                        object
legend                      object
shape_len                   object
geometry                  geometry
dtype: object

TypeError: string indices must be integers, not 'str'

In [33]:
cta_lines.head()

,:id,:version,:created_at,:updated_at,lines,description,type,legend,shape_len,geometry
0,row-jfz4-3j8m~bhsb,rv-ucfc-iy3g~j5vu,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Orange, Pink, Purple (Express)",Tower 12 to Library,Elevated or At Grade,ML,647.793224715,"MULTILINESTRING ((358531.229 578671.331, 35858..."
1,row-j8yp-4buv.s655,rv-yw5q_s4wj.dk6g,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Purple",Subway Portal B to Sedgwick,Elevated or At Grade,BR,4303.79492895,"MULTILINESTRING ((357580.312 582384.742, 35754..."
2,row-m5nn~7ffa-s9eu,rv-ky69_kik2~4emh,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,Yellow Line,Oakton-Skokie to Howard,Elevated or At Grade,YL,21195.44443,"MULTILINESTRING ((348540.536 595177.981, 34854..."
3,row-m9xp_txdh-c9wu,rv-hfga~uzwm-4f37,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Green, Orange, Pink, Purple (Exp)",Washington/Wabash to Adams/Wabash,Elevated or At Grade,ML,1352.12055387,"MULTILINESTRING ((358704.1 578961.761, 358696...."
4,row-bf34.b8eu-gbg7,rv-urkc.8hjb_b78p,2024-07-23 22:38:55.007000+00:00,2024-07-23 22:38:55.007000+00:00,"Brown, Green, Orange, Pink, Purple (Exp)",State/Lake to Washington/Wabash,Elevated or At Grade,ML,1317.60492405,"MULTILINESTRING ((358693.16 579373.743, 358685..."
